# `Hough lines` and `Hough Circles`
The Hough Line Transform is a computer vision technique used to detect straight
lines in an image. It is widely used in various applications such as lane detection
in autonomous vehicles, edge detection, and shape analysis.

The Hough Circle Transform is a computer vision technique used to detect circles
in an image. It is widely used in various applications such as face detection,
object detection, and tracking.


 Probabilistic Hough Transform (PHT) is a modified version of the Hough
Line Transform that is more efficient and accurate for detecting lines in an
image.

In [ ]:
import cv2
import numpy as np

img = np.zeros((200,300,3),np.uint8)
# drawing a line onto that img

cv2.line(
    img,
    (0,10),
    (100,40),
    (0,255,0)
)
cv2.line(
    img,
    (10,10),
    (10,100),
    (0,255,0),
    2
)
cv2.imshow("Img",img)

grayed = cv2.cvtColor(img,cv2.COLOR_BGR2GRAY)
edges = cv2.Canny(
    grayed,
    20,
    150,
    apertureSize=3
)
# applying hough line transformation
lines = cv2.HoughLinesP(
    edges,
    rho=1,
    theta=np.pi/180,
    threshold = 20,
    minLineLength=2,
    maxLineGap=10
)
# print(lines)
# Draw detected lines
for line in lines:
    # rho,theta = line[0]
    # a = np.cos(theta)
    # b = np.sin(theta)
    
    # x0 = a* rho
    # y0 = b*rho
    
    # x1 = int(x0+1000*(-b))
    # y1 = int(y0+1000*(a))
    # x2 = int(x0-1000*(-b))
    # y2 = int(y0-1000*(a))
    x1, y1, x2, y2 = line[0]

    cv2.line(
        img,
        (x1,y1),
        (x2,y2),
        (0,0,255),
        2
    )
cv2.imshow("Result",img)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [1]:
import cv2 # Imports OpenCV
# frame = cv2.imread("image.jpg") # Loads the color source image

cap = cv2.VideoCapture(0)
while 1:
    ret, frame = cap.read()
        
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY) # Converts the frame to intensi
    gray = cv2.medianBlur(gray, 5) # Reduces isolated noise before thresholding
    _, mask = cv2.threshold(gray, 150, 255, cv2.THRESH_BINARY_INV) # Creates a bina
    # mask = cv2.erode(mask, None, iterations=2) # Removes small white noise
    # mask = cv2.dilate(mask, None, iterations=2) # Restores and connects foreground
    mask = cv2.morphologyEx(mask,cv2.MORPH_OPEN,(2,2))
    foreground = cv2.bitwise_and(frame, frame, mask=mask) # Keeps only selected pix
    cv2.imshow("Foreground", foreground) # Displays the result
    if cv2.waitKey(1) & 0xFF == ord('q'): # Exits the program if q is pressed
        break
cap.release()
cv2.destroyAllWindows()

In [13]:
import cv2

img = cv2.resize(cv2.imread("..\\fol\\2dshapes.png"),(0,0),fx=0.5,fy=0.5)

gray = cv2.cvtColor(img,cv2.COLOR_BGR2GRAY)
edges  = cv2.Canny(gray,50,150,apertureSize = 3)
contours,_ = cv2.findContours(
    edges,
    cv2.RETR_EXTERNAL,
    cv2.CHAIN_APPROX_SIMPLE
)

for cnt in contours:
    area = cv2.contourArea(cnt)
    if area<100: continue
    perimeter = cv2.arcLength(cnt,True)
    approx = cv2.approxPolyDP(cnt,0.02*perimeter,True)
    print(perimeter)
    x,y,w,h = cv2.boundingRect(approx)
    if len(approx) == 3:
        label = 'Triangle'
    elif len(approx) == 4:
        label = 'Square'
    elif len(approx) == 5:
        label = 'Pentagon'
    elif len(approx) == 6:
        label = 'Hexagon'
    else:
        label = 'Circle'
    cv2.drawContours(img,[approx],0,(0,255,0),2)
    cv2.putText(img,label,(x,y),cv2.FONT_HERSHEY_SIMPLEX,0.5,(0,255,0),2)
    
cv2.imshow("Image",img)
cv2.waitKey(0)  
cv2.destroyAllWindows()

131.49747371673584
59.21320295333862
239.1370838880539
246.7939373254776
352.936070561409
245.62236428260803
251.62236309051514
257.13708305358887
240.61017155647278
372.02438521385193
432.3675310611725
336.56854152679443
216.3919162750244
234.7939372062683
236.40916121006012
399.65685415267944
280.0
215.82337498664856
113.01219260692596
50.82842707633972
65.0710676908493
113.74011385440826
98.18376529216766
94.62741649150848
96.76955163478851
105.0121922492981
98.76955151557922
162.5269112586975
72.28427052497864
99.35533821582794
124.38477575778961
153.29646277427673
90.97056245803833
159.15432786941528
94.97056245803833
132.56854248046875


In [35]:
import cv2 # Imports OpenCV
capture = cv2.VideoCapture(0) # Opens the video
subtractor = cv2.createBackgroundSubtractorKNN( # Creates the KNN model
    history=200, # Argument
    dist2Threshold=400, # Argument
    detectShadows=True, # Argument
)
while capture.isOpened(): # Runs while the source is available
    success, frame = capture.read() # Reads one frame
    if not success: # Handles the end of the source
        break # Leaves the loop
    mask = subtractor.apply(frame) # Updates the KNN background and creates a m
    
    
    back = cv2.bitwise_and(frame,frame,mask=mask) # Creates a background estimate

    cv2.imshow("KNN foreground", mask) # Displays the foreground estimate
    cv2.imshow("KNN background", back) # Displays the background estimate
    if cv2.waitKey(30) & 0xFF == ord("q"): # Checks for q
        break # Stops playback
capture.release() # Releases the source
cv2.destroyAllWindows()

In [21]:
# detecting road lanes

import cv2
import numpy as np

cap = cv2.VideoCapture(r"../fol/test_video.mp4")

while cap.isOpened():
    ret,frame = cap.read()
    if not ret:
        break
    grayed = cv2.cvtColor(frame,cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(grayed,100,200)
    lines = cv2.HoughLinesP(edges,1,np.pi/180,50,minLineLength=50,maxLineGap=10)
    print(lines)
    for x1,y1,x2,y2 in lines:
        cv2.line(frame,(x1,y1),(x2,y2),(0,255,0),2)
    cv2.imshow('frame',frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

[[ 763  467  933  626]
 [   1  552  301  412]
 [   0  552  496  320]
 [ 683  396 1006  719]
 [ 633  345  769  481]
 [ 745  459 1004  718]
 [ 397  580  500  408]
 [ 356  690  478  460]
 [ 242  443  494  325]
 [ 604  313  777  480]
 [ 357  649  444  504]
 [ 104  509  281  426]
 [ 442  504  507  395]
 [ 523  205  595  193]
 [ 111  265  204  265]
 [   0  559  155  486]
 [1134   98 1190   88]
 [ 302  414  502  320]
 [ 949  641 1000  692]
 [ 812  151  903  116]
 [ 453  296  516  291]
 [ 290  416  504  316]
 [ 417  219  473  209]
 [ 117  323  200  320]
 [ 827  146  890  122]
 [ 714  418  821  521]
 [ 165  305  216  302]
 [ 838  538  953  646]
 [ 605  321  657  368]]
[[ 683  396 1006  719]
 [   0  552  493  322]
 [ 595  310  667  378]
 [ 692  397  861  560]
 [ 267  431  497  323]
 [ 344  711  396  614]
 [  58  531  273  430]
 [ 388  598  487  428]
 [ 864  560 1004  695]
 [ 651  358  782  485]
 [ 654  366  769  481]
 [ 111  265  204  265]
 [ 745  459 1004  718]
 [ 779  480  889  586]
 [ 357  68

In [36]:
# corner detection

import cv2

img = cv2.imread("..\\fol\\Screenshot 2026-10-05 154513.png",0)
op = img.copy()

corners = cv2.cornerHarris(
    img,2,3,0.04
)
corners = cv2.dilate(corners, None)
threshold = 0.01 * corners.max()
op[corners > threshold]= [255]
cv2.imshow("Corners",op)
cv2.waitKey(0)
cv2.destroyAllWindows()


In [ ]:
import cv2
import numpy as np

# --------------------------------
# Start webcam
# --------------------------------
cap = cv2.VideoCapture(0)

# Read first frame
ret, frame = cap.read()

if not ret:
    print("Could not open camera")
    exit()

# --------------------------------
# Select object manually
# --------------------------------
x, y, w, h = cv2.selectROI(
    "Select Object",
    frame,
    False
)

cv2.destroyWindow("Select Object")

# Tracking window
track_window = (x, y, w, h)

# --------------------------------
# Convert first frame to HSV
# --------------------------------
hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

# --------------------------------
# Get HSV values of selected object
# --------------------------------

roi = hsv[y:y+h, x:x+w]

# --------------------------------
# Create mask
# Ignore very dark and low-saturation pixels
# --------------------------------
mask = cv2.inRange(
    roi,
    np.array((0, 60, 32)),
    np.array((180, 255, 255))
)

# --------------------------------
# Calculate histogram
# --------------------------------
roi_hist = cv2.calcHist(
    [roi],
    [0],
    mask,
    [180],
    [0, 180]
)

# Normalize histogram
cv2.normalize(
    roi_hist,
    roi_hist,
    0,
    255,
    cv2.NORM_MINMAX
)

# --------------------------------
# Start tracking
# --------------------------------
while True:

    ret, frame = cap.read()

    if not ret:
        break

    # Convert current frame to HSV
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

    # --------------------------------
    # Back projection
    # --------------------------------
    back_project = cv2.calcBackProject(
        [hsv],
        [0],
        roi_hist,
        [0, 180],
        1
    )

    # --------------------------------
    # Mean Shift
    # --------------------------------
    ret, track_window = cv2.meanShift(
        back_project,
        track_window,
        (
            cv2.TERM_CRITERIA_EPS |
            cv2.TERM_CRITERIA_COUNT,
            10,
            1
        )
    )

    # --------------------------------
    # Draw tracking rectangle
    # --------------------------------
    x, y, w, h = track_window

    cv2.rectangle(
        frame,
        (x, y),
        (x + w, y + h),
        (0, 255, 0),
        2
    )
    # Display
    cv2.imshow("Mean Shift Tracking", frame)
    cv2.imshow("Back Projection", back_project)

    # ESC to exit
    if cv2.waitKey(30) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()


In [2]:
import cv2  # Imports OpenCV
import numpy as np  # Imports NumPy

capture = cv2.VideoCapture(0)  # Opens the tracking video
success, first = capture.read()  # Reads the initial frame
if not success:  # Checks the input
    raise RuntimeError("Could not read object.mp4")  # Reports the failure
ret,frame = cap.read()
x,y,w,h = cv2.selectROI(
    'SELECTROI',
    frame,
    False
)
window = (x,y,w,h)
cv2.destroyWindow('SELECTROI')
# x, y, w, h = window  # Unpacks the initial window
hsv_first = cv2.cvtColor(first, cv2.COLOR_BGR2HSV)  # Converts the first frame to HSV
roi = hsv_first[y:y + h, x:x + w]  # Extracts the initial ROI
roi_mask = cv2.inRange(roi, np.array([0, 30, 20]), np.array([180, 255, 255]))  # Builds a valid-pixel mask
histogram = cv2.calcHist([roi], [0], roi_mask, [180], [0, 180])  # Computes the hue histogram
cv2.normalize(histogram, histogram, 0, 255, cv2.NORM_MINMAX)  # Normalizes the histogram
term = (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 1)  # Sets search termination criteria
while capture.isOpened():  # Processes each frame
    success, frame = capture.read()  # Reads a frame
    if not success:  # Detects end-of-file
        break  # Stops tracking
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)  # Converts the frame to HSV
    back_projection = cv2.calcBackProject([hsv], [0], histogram, [0, 180], 1)  # Builds the probability image
    rotated_rect, window = cv2.CamShift(  # Adapts position, size, and orientation
        back_projection,  # Argument
        window,  # Argument
        term,  # Argument
    )
    points = cv2.boxPoints(rotated_rect).astype(np.int32)  # Converts the rotated box to corner points
    cv2.polylines(frame, [points], True, (0, 255, 0), 2)  # Draws the rotated tracking box
    cv2.imshow("CAMShift", frame)  # Displays the tracking result
    if cv2.waitKey(30) & 0xFF == ord("q"):  # Checks for q
        break  # Stops tracking
capture.release()  # Releases the video
cv2.destroyAllWindows()  # Closes the window


In [20]:
# Match template

import cv2
import numpy as np

cap = cv2.VideoCapture(0)
ret,frame = cap.read()
x,y,w,h = cv2.selectROI('frame',frame)
object = frame[y:y+h,x:x+w]

while True:
    ret,frame = cap.read()
    match = cv2.matchTemplate(
        frame,
        object,
        cv2.TM_CCORR_NORMED
    )
    min_val, max_val, min_loc, max_loc = cv2.minMaxLoc(match)
    if max_val>0.3:
        x,y = max_loc
    
    cv2.rectangle(frame,max_loc,(x+w,y+h),(0,0,255),2)
    cv2.imshow('frame',frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

In [24]:
import cv2
from matplotlib.transforms import ScaledTranslation
import numpy as np

face_cascade = cv2.CascadeClassifier(
 'haarcascade_frontalface_default.xml'   
)

cap = cv2.VideoCapture(0)
while True:
    ret,frame = cap.read()
    if not ret: break
    grayed = cv2.cvtColor(frame,cv2.COLOR_BGR2GRAY)
    
    faces = face_cascade.detectMultiScale(grayed,scaleFactor=1.2,minNeighbors=5,minSize = (30,30))
    for (x,y,w,h) in faces:
        cv2.rectangle(
            frame,
            (x,y),
            (x+w,y+h),
            (0,255,0),
            2
        )
    cv2.imshow('live',frame[:,::-1])
    if cv2.waitKey(1) == ord('q'): break
cap.release()
cv2.destroyAllWindows()    



In [4]:
cv2.destroyAllWindows()